# EE-411 — Fundamentals of Inference and Learning

## Homework 2 — Optimization of a CNN model

The task of this homework is to optimize a CNN model for CIFAR-100. You are free to define the model architecture and training procedure. The only constraints are:

- It must be a `torch.nn.Module` object.
- The number of trained parameters must be less than 1 million.
- The test dataset must not be used for any step of training.
- The final training notebook should run on Google Colab in approximately 1 hour at most.
- Do not modify the random seeds, as they are needed for reproducibility.

For grading, you must use the `evaluate` function defined below. It takes a model as input and prints the test accuracy.

As a guideline, you are expected to **discuss** and motivate your choices regarding:

- Model architecture
- Hyperparameters (learning rate, batch size, etc.)
- Regularization methods
- Optimizer
- Validation scheme

Code without an explanation of the choices will not be accepted. Test accuracy is not the only measure of success for this homework.

Most of the training process is randomized. Store your model's weights after training and load them before evaluation. **Hand in the filled in HW2_practical_ex.ipynb file together with the neural_net.pt file of your trained network.**

### Example

#### Loading packages and libraries

In [2]:
import random
import numpy as np
import torch
import torchvision


# Fix all random seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# For full determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Import the best device available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

# load the data
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=torchvision.transforms.ToTensor())

Using device: cuda


  1%|          | 1.21M/169M [00:12<28:56, 96.6kB/s]


KeyboardInterrupt: 

In [6]:
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=torchvision.transforms.ToTensor())

def evaluate(model):
    params_count = sum(p.numel() for p in model.parameters())
    print('The model has {} parameters'.format(params_count))

    if params_count >= int(1e6):
        print('The model has too many parameters! Not allowed to evaluate.')
        return

    model = model.to(device)
    model.eval()
    correct = 0
    total = 0

    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


    # print in bold red in a notebook
    print('\033[1m\033[91mAccuracy on the test set: {}%\033[0m'.format(100 * correct / total))


#### Example of a simple CNN model

In [7]:
class TinyNet(torch.nn.Module):
    def __init__(self):
        super(TinyNet, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = torch.nn.Linear(8*8*64, 128)
        self.fc2 = torch.nn.Linear(128, 100)

    def forward(self, x):
        x = torch.nn.functional.relu(self.conv1(x))
        x = torch.nn.functional.max_pool2d(x, 2)
        x = torch.nn.functional.relu(self.conv2(x))
        x = torch.nn.functional.max_pool2d(x, 2)
        x = x.view(-1, 8*8*64)
        x = torch.nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print("Model parameters: ", sum(p.numel() for p in TinyNet().parameters()))

Model parameters:  556708


#### Example of basic training

In [8]:

model = TinyNet()
model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters())

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
for epoch in range(10):
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1, 10, loss.item()))


Epoch [1/10], Loss: 4.6138
Epoch [2/10], Loss: 4.5808
Epoch [3/10], Loss: 4.5756
Epoch [4/10], Loss: 4.5898
Epoch [5/10], Loss: 4.5960
Epoch [6/10], Loss: 4.5979
Epoch [7/10], Loss: 4.6144
Epoch [8/10], Loss: 4.5326
Epoch [9/10], Loss: 4.5009
Epoch [10/10], Loss: 4.3965


In [9]:
# save the model on a file
torch.save(model.state_dict(), 'neural_net.pt')

loaded_model = TinyNet()
loaded_model.load_state_dict(torch.load('neural_net.pt', weights_only=True))
evaluate(loaded_model)

The model has 556708 parameters
Accuracy on the test set: 3.35%


In [10]:
from google.colab import files

files.download('neural_net.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import os

print(os.path.abspath('neural_net.pt'))
print(os.path.getsize('neural_net.pt'), 'bytes')

/content/neural_net.pt
2230519 bytes


## Optimized CNN model for CIFAR-100

The model is build following these steps :
1. Creation of a training and a validation set extract from the given dataset
2.
3.
4.

In [1]:
# Importations and librairies
import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms

In [2]:
# Fix all random seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# For full determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Import the best device available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

# load the data
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=torchvision.transforms.ToTensor())

Using device: cuda


100%|██████████| 169M/169M [25:31<00:00, 110kB/s]  


In [4]:
test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=torchvision.transforms.ToTensor())

def evaluate(model):
    params_count = sum(p.numel() for p in model.parameters())
    print('The model has {} parameters'.format(params_count))

    if params_count >= int(1e6):
        print('The model has too many parameters! Not allowed to evaluate.')
        return

    model = model.to(device)
    model.eval()
    correct = 0
    total = 0

    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


    # print in bold red in a notebook
    print('\033[1m\033[91mAccuracy on the test set: {}%\033[0m'.format(100 * correct / total))


### Step 1 - Creation of a training and a validation set

In [6]:
# 1. Data preparation and validation split

# Load the original CIFAR-100 training dataset
full_train_dataset = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=True,
    transform=torchvision.transforms.ToTensor()
)

# Split into 90% training and 10% validation
my_train_dataset, my_val_dataset = random_split(
    full_train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# Create data loaders
my_train_loader = DataLoader(
    my_train_dataset,
    batch_size=64,
    shuffle=True
)

my_val_loader = DataLoader(
    my_val_dataset,
    batch_size=64,
    shuffle=False
)

print("Training images:", len(my_train_dataset))
print("Validation images:", len(my_val_dataset))


Training images: 45000
Validation images: 5000


In [5]:

# Experiment B - Data Augmentation

import torch
import torchvision
from torchvision import transforms
from torch.utils.data import random_split, Subset, DataLoader

# 1. Define transformations for training
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

# 2. Define transformations for validation
val_transform = transforms.ToTensor()

# 3. Load the original CIFAR-100 training dataset
full_train_dataset = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=True,
    transform=val_transform
)

# 4. Recreate the same split as Experiment A
train_split, val_split = random_split(
    full_train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# 5. Save the indices
train_indices = train_split.indices
val_indices = val_split.indices

# 6. Create a dataset with data augmentation
train_data_aug = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=False,
    transform=train_transform
)

# 7. Create a dataset without data augmentation
val_data_aug = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=False,
    transform=val_transform
)

# 8. Apply the same indices to both datasets
my_train_dataset = Subset(
    train_data_aug,
    train_indices
)

my_val_dataset = Subset(
    val_data_aug,
    val_indices
)

# 9. Create DataLoaders
my_train_loader = DataLoader(
    my_train_dataset,
    batch_size=64,
    shuffle=True
)

my_val_loader = DataLoader(
    my_val_dataset,
    batch_size=64,
    shuffle=False
)

# 10. Verify dataset sizes
print("Training images:", len(my_train_dataset))
print("Validation images:", len(my_val_dataset))
print("Data augmentation applied successfully!")


Training images: 45000
Validation images: 5000
Data augmentation applied successfully!


### Step 2 - CNN Architecture

In [11]:
# Step 2 - CNN Architecture

import torch.nn.functional as F

class MyCNN(nn.Module):
    def __init__(self):
        super(MyCNN, self).__init__()

        # First convolutional block
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Second convolutional block
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Third convolutional block
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Global average pooling
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Classification layer
        self.fc = nn.Linear(128, 100)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.bn1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.bn2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.bn3(self.conv3(x))), 2)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

# Create our model
my_model = MyCNN().to(device)

# Count parameters
params_count = sum(p.numel() for p in my_model.parameters())

print("Number of parameters:", params_count)
print("Model architecture:")
print(my_model)


Number of parameters: 106596
Model architecture:
MyCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (avgpool): AdaptiveAvgPool2d(output_size=(1, 1))
  (fc): Linear(in_features=128, out_features=100, bias=True)
)


In [ ]:

# Experiment C - Deeper CNN

class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(128, 192, 3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.Conv2d(192, 192, 3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(192, 100)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# Create a new model
my_model = BetterCNN().to(device)

params_count = sum(
    p.numel() for p in my_model.parameters()
)

print("Number of parameters:", params_count)
assert params_count < 1_000_000


Number of parameters: 834340


In [6]:

# Experiment G - Bigger CNN

import torch.nn as nn

class BiggerCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(128, 200, 3, padding=1),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.Conv2d(200, 200, 3, padding=1),
            nn.BatchNorm2d(200),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(200, 100)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


my_model = BiggerCNN().to(device)

params_count = sum(p.numel() for p in my_model.parameters())

print("Number of parameters:", params_count)

assert params_count < 1_000_000, "Too many parameters!"


Number of parameters: 872628


### Step 3 - Training Configuration

In [7]:

# MixUp augmentation

import numpy as np

def mixup_data(images, labels, alpha=0.2):

    lam = np.random.beta(alpha, alpha)

    batch_size = images.size(0)

    index = torch.randperm(
        batch_size,
        device=images.device
    )

    mixed_images = (
        lam * images
        + (1 - lam) * images[index]
    )

    labels_a = labels
    labels_b = labels[index]

    return mixed_images, labels_a, labels_b, lam


# CutMix augmentation

import torch

def cutmix_data(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)

    batch_size, _, height, width = images.shape
    index = torch.randperm(batch_size, device=images.device)

    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(width * cut_ratio)
    cut_h = int(height * cut_ratio)

    cx = np.random.randint(width)
    cy = np.random.randint(height)

    x1 = np.clip(cx - cut_w // 2, 0, width)
    x2 = np.clip(cx + cut_w // 2, 0, width)
    y1 = np.clip(cy - cut_h // 2, 0, height)
    y2 = np.clip(cy + cut_h // 2, 0, height)

    mixed_images = images.clone()

    mixed_images[:, :, y1:y2, x1:x2] = (
        images[index, :, y1:y2, x1:x2]
    )

    # Correct lambda based on actual patch area
    lam = 1.0 - ((x2 - x1) * (y2 - y1)) / (width * height)

    return mixed_images, labels, labels[index], lam


# Randomly choose MixUp or CutMix

def apply_mix_augmentation(images, labels):

    if np.random.rand() < 0.5:
        return mixup_data(
            images, labels, alpha=0.2
        )
    else:
        return cutmix_data(
            images, labels, alpha=1.0
        )


In [12]:
# Step 3 - Training configuration

import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    my_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

num_epochs = 20

print("Loss function:", criterion)
print("Optimizer: AdamW")
print("Learning rate:", 0.001)
print("Epochs:", num_epochs)

Loss function: CrossEntropyLoss()
Optimizer: AdamW
Learning rate: 0.001
Epochs: 20


In [8]:

# Training configuration - Experiment C

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    my_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

num_epochs = 35

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs
)


In [8]:

# Experiment E - Label smoothing

# Add label smoothing
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

# Keep the same optimizer as Experiment C
optimizer = torch.optim.AdamW(
    my_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

# Same training duration as Experiment C
num_epochs = 35

# Same learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs
)

print("Experiment E initialized")
print("Label smoothing:", 0.1)
print("Epochs:", num_epochs)
print("Parameters:", sum(p.numel() for p in my_model.parameters()))

Experiment E initialized
Label smoothing: 0.1
Epochs: 35
Parameters: 834340


In [26]:

# Experiment G - Training configuration
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    my_model.parameters(),
    lr=0.001,
    weight_decay=0.0001
)

num_epochs = 35

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs
)

print("Experiment G ready!")


Experiment G ready!


### Step 4 - Training and Validation

In [16]:

# Step 4 - Training and validation

import time
import copy

start_time = time.time()

train_losses = []
val_losses = []
val_accuracies = []

best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):

    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = my_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    train_loss = running_loss / total_train
    train_losses.append(train_loss)

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = my_model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    # Save the best model weights
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

# Restore the best model
my_model.load_state_dict(best_model_weights)

# Save the model
torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time

print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")
print("Best model saved to neural_net.pt")


Epoch [1/35] Train Loss: 3.6190 | Val Loss: 3.5811 | Val Accuracy: 17.18%
Epoch [2/35] Train Loss: 2.8335 | Val Loss: 2.9413 | Val Accuracy: 26.08%
Epoch [3/35] Train Loss: 2.4045 | Val Loss: 2.5309 | Val Accuracy: 34.86%
Epoch [4/35] Train Loss: 2.1097 | Val Loss: 2.5023 | Val Accuracy: 38.14%
Epoch [5/35] Train Loss: 1.9046 | Val Loss: 2.3365 | Val Accuracy: 39.62%
Epoch [6/35] Train Loss: 1.7516 | Val Loss: 2.0501 | Val Accuracy: 46.26%
Epoch [7/35] Train Loss: 1.6247 | Val Loss: 1.7951 | Val Accuracy: 50.74%
Epoch [8/35] Train Loss: 1.5146 | Val Loss: 1.7102 | Val Accuracy: 52.08%
Epoch [9/35] Train Loss: 1.4215 | Val Loss: 1.7680 | Val Accuracy: 53.46%
Epoch [10/35] Train Loss: 1.3442 | Val Loss: 1.6758 | Val Accuracy: 54.98%
Epoch [11/35] Train Loss: 1.2668 | Val Loss: 1.5844 | Val Accuracy: 56.72%
Epoch [12/35] Train Loss: 1.2002 | Val Loss: 1.6641 | Val Accuracy: 55.82%
Epoch [13/35] Train Loss: 1.1392 | Val Loss: 1.6180 | Val Accuracy: 56.68%
Epoch [14/35] Train Loss: 1.0832 |

In [27]:


# Step 4 - Training and validation

import time
import copy

start_time = time.time()

train_losses = []
val_losses = []
val_accuracies = []

best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):

    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Apply MixUp during training
        images, labels_a, labels_b, lam = mixup_data(
        images,
        labels,
        alpha=0.2
        )

        outputs = my_model(images)

        loss = (
        lam * criterion(outputs, labels_a)
        + (1 - lam) * criterion(outputs, labels_b)
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    train_loss = running_loss / total_train
    train_losses.append(train_loss)

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = my_model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    # Save the best model weights
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

# Restore the best model
my_model.load_state_dict(best_model_weights)

# Save the model
torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time

print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")
print("Best model saved to neural_net.pt")



Epoch [1/35] Train Loss: 3.8825 | Val Loss: 3.9210 | Val Accuracy: 11.40%
Epoch [2/35] Train Loss: 3.2428 | Val Loss: 3.0112 | Val Accuracy: 24.68%
Epoch [3/35] Train Loss: 2.9181 | Val Loss: 2.5225 | Val Accuracy: 34.44%
Epoch [4/35] Train Loss: 2.6629 | Val Loss: 2.3058 | Val Accuracy: 38.88%
Epoch [5/35] Train Loss: 2.5429 | Val Loss: 2.1009 | Val Accuracy: 44.04%
Epoch [6/35] Train Loss: 2.3470 | Val Loss: 1.9310 | Val Accuracy: 47.64%
Epoch [7/35] Train Loss: 2.2721 | Val Loss: 1.7721 | Val Accuracy: 50.76%
Epoch [8/35] Train Loss: 2.1678 | Val Loss: 1.7151 | Val Accuracy: 52.62%
Epoch [9/35] Train Loss: 2.0949 | Val Loss: 1.7065 | Val Accuracy: 53.02%
Epoch [10/35] Train Loss: 2.0638 | Val Loss: 1.8260 | Val Accuracy: 50.76%
Epoch [11/35] Train Loss: 2.0116 | Val Loss: 1.6773 | Val Accuracy: 55.00%
Epoch [12/35] Train Loss: 1.9630 | Val Loss: 1.5641 | Val Accuracy: 56.48%
Epoch [13/35] Train Loss: 1.8471 | Val Loss: 1.5333 | Val Accuracy: 56.80%
Epoch [14/35] Train Loss: 1.8453 |

In [10]:

import shutil

shutil.copyfile("neural_net.pt", "model_C.pt")

print("Experiment C saved successfully!")

Experiment C saved successfully!


In [9]:

# Experiment I - Training with MixUp OR CutMix

import time
import copy

start_time = time.time()

train_losses = []
val_losses = []
val_accuracies = []

best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):

    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Randomly apply MixUp OR CutMix
        images, labels_a, labels_b, lam = apply_mix_augmentation(
            images, labels
        )

        outputs = my_model(images)

        loss = (
            lam * criterion(outputs, labels_a)
            + (1 - lam) * criterion(outputs, labels_b)
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    train_loss = running_loss / total_train
    train_losses.append(train_loss)

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = my_model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            predicted = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    # Update scheduler once per epoch
    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )

# Restore best weights
my_model.load_state_dict(best_model_weights)

# Save this experiment separately
torch.save(my_model.state_dict(), "model_I.pt")

elapsed_time = time.time() - start_time

print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")
print("Best model saved to model_I.pt")


Epoch [1/35] Train Loss: 4.0345 | Val Loss: 3.4605 | Val Accuracy: 17.02%
Epoch [2/35] Train Loss: 3.5912 | Val Loss: 3.0564 | Val Accuracy: 23.70%
Epoch [3/35] Train Loss: 3.3162 | Val Loss: 3.1538 | Val Accuracy: 24.42%
Epoch [4/35] Train Loss: 3.1289 | Val Loss: 2.5737 | Val Accuracy: 35.00%
Epoch [5/35] Train Loss: 2.9428 | Val Loss: 2.3897 | Val Accuracy: 38.42%
Epoch [6/35] Train Loss: 2.8850 | Val Loss: 2.2240 | Val Accuracy: 42.46%
Epoch [7/35] Train Loss: 2.8040 | Val Loss: 2.0724 | Val Accuracy: 45.12%
Epoch [8/35] Train Loss: 2.7127 | Val Loss: 1.9512 | Val Accuracy: 46.68%
Epoch [9/35] Train Loss: 2.6623 | Val Loss: 1.9834 | Val Accuracy: 46.70%
Epoch [10/35] Train Loss: 2.5608 | Val Loss: 1.8608 | Val Accuracy: 50.80%
Epoch [11/35] Train Loss: 2.5688 | Val Loss: 1.7633 | Val Accuracy: 52.70%
Epoch [12/35] Train Loss: 2.4329 | Val Loss: 1.7387 | Val Accuracy: 53.08%
Epoch [13/35] Train Loss: 2.4351 | Val Loss: 1.7057 | Val Accuracy: 53.86%
Epoch [14/35] Train Loss: 2.3923 |

In [16]:
import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
import time
import copy

# 1. Fixation des seeds pour la reproductibilité stricte
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

# 2. Préparation des données avec AutoAugment
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10), # Augmentation forte
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)) 
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

# Chargement
train_data_aug = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=train_transform)
val_data_norm = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=val_transform)

# Split 90% / 10%
full_train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transforms.ToTensor())
my_train_dataset, my_val_dataset = random_split(full_train_dataset, [45000, 5000], generator=torch.Generator().manual_seed(42))

my_train_dataset = Subset(train_data_aug, my_train_dataset.indices)
my_val_dataset = Subset(val_data_norm, my_val_dataset.indices)

# DataLoaders
batch_size = 128
my_train_loader = DataLoader(my_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
my_val_loader = DataLoader(my_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

Using device: cuda


In [17]:
# Module Squeeze-and-Excitation
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

# Bloc Résiduel avec SE
class SEResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class AdvancedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.prep = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # Architecture parfaitement dimensionnée pour frôler le million
        self.layer1 = SEResidualBlock(64, 64, stride=1)     # -> Sortie 64 canaux
        self.layer2 = SEResidualBlock(64, 128, stride=2)    # -> Sortie 128 canaux
        self.layer3 = SEResidualBlock(128, 128, stride=2)   # -> Sortie 128 canaux
        self.layer4 = SEResidualBlock(128, 128, stride=1)   # -> Sortie 128 canaux (L'ajout parfait)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 100)
        )

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.classifier(x)

# Instanciation et vérification
my_model = AdvancedCNN().to(device)
params_count = sum(p.numel() for p in my_model.parameters())

print(f"Number of parameters: {params_count}")
assert params_count < 1000000, "Le modèle a trop de paramètres !"

Number of parameters: 933028


In [18]:
# Utilisation du Label Smoothing pour réduire le surapprentissage
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# L'optimiseur (weight_decay augmenté pour une meilleure régularisation)
optimizer = torch.optim.AdamW(my_model.parameters(), lr=0.005, weight_decay=0.05) 

# Plus d'époques pour laisser AutoAugment et OneCycleLR faire leur effet
num_epochs = 150

# OneCycleLR change le learning rate à *chaque batch* (et non à chaque époque)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.005,
    steps_per_epoch=len(my_train_loader),
    epochs=num_epochs,
    pct_start=0.3 # Atteint le max_lr à 30% de l'entraînement
)

start_time = time.time()
best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):
    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = my_model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        # Le scheduler OneCycleLR s'update à chaque itération (batch)
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    train_loss = running_loss / total_train

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = my_model(images)
            # On calcule la loss sans label smoothing pour la validation (plus représentatif)
            val_loss_criterion = nn.CrossEntropyLoss()
            loss = val_loss_criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    # Sauvegarde du meilleur modèle
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    # Affichage simplifié toutes les 5 époques (ou si c'est la meilleure)
    if (epoch + 1) % 5 == 0 or val_accuracy == best_val_accuracy:
        print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

# Restauration du meilleur modèle et sauvegarde finale
my_model.load_state_dict(best_model_weights)
torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time
print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")

Epoch [1/150] Train Loss: 4.3942 | Val Loss: 3.9046 | Val Acc: 10.78%
Epoch [2/150] Train Loss: 4.1584 | Val Loss: 3.6681 | Val Acc: 14.46%
Epoch [3/150] Train Loss: 3.9778 | Val Loss: 3.3454 | Val Acc: 20.90%
Epoch [4/150] Train Loss: 3.8176 | Val Loss: 3.1447 | Val Acc: 24.96%
Epoch [5/150] Train Loss: 3.6840 | Val Loss: 3.0006 | Val Acc: 25.92%
Epoch [6/150] Train Loss: 3.5834 | Val Loss: 2.8450 | Val Acc: 29.48%
Epoch [7/150] Train Loss: 3.4778 | Val Loss: 2.7305 | Val Acc: 30.92%
Epoch [8/150] Train Loss: 3.3810 | Val Loss: 2.6392 | Val Acc: 31.96%
Epoch [9/150] Train Loss: 3.2884 | Val Loss: 2.4742 | Val Acc: 37.16%
Epoch [10/150] Train Loss: 3.1988 | Val Loss: 2.4934 | Val Acc: 34.92%
Epoch [11/150] Train Loss: 3.1324 | Val Loss: 2.3636 | Val Acc: 38.48%
Epoch [12/150] Train Loss: 3.0609 | Val Loss: 2.3426 | Val Acc: 39.06%
Epoch [13/150] Train Loss: 3.0075 | Val Loss: 2.2112 | Val Acc: 39.68%
Epoch [14/150] Train Loss: 2.9427 | Val Loss: 2.1212 | Val Acc: 44.10%
Epoch [15/150] 

In [23]:
import random
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
import time
import copy

# Fixation des seeds
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

# Transformation avec AutoAugment
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)) 
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

train_data_aug = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=train_transform)
val_data_norm = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=val_transform)

full_train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transforms.ToTensor())
my_train_dataset, my_val_dataset = random_split(full_train_dataset, [45000, 5000], generator=torch.Generator().manual_seed(42))

my_train_dataset = Subset(train_data_aug, my_train_dataset.indices)
my_val_dataset = Subset(val_data_norm, my_val_dataset.indices)

# Batch size augmenté à 256 pour accélérer les époques
batch_size = 256
my_train_loader = DataLoader(my_train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
my_val_loader = DataLoader(my_val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

Using device: cuda


In [24]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class SEResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEBlock(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class AdvancedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.prep = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        self.layer1 = SEResidualBlock(64, 64, stride=1)     
        self.layer2 = SEResidualBlock(64, 128, stride=2)    
        self.layer3 = SEResidualBlock(128, 128, stride=2)   
        self.layer4 = SEResidualBlock(128, 128, stride=1)   
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.1),
            nn.Linear(128, 100)
        )

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.classifier(x)

my_model = AdvancedCNN().to(device)
params_count = sum(p.numel() for p in my_model.parameters())
print(f"Number of parameters: {params_count}")
assert params_count < 1000000, "Le modèle a trop de paramètres !"

Number of parameters: 933028


In [19]:
from torchvision.transforms import v2

# 1. Initialisation de CutMix et Mixup pour 100 classes
cutmix = v2.CutMix(alpha=1.0, num_classes=100)
mixup = v2.MixUp(alpha=0.2, num_classes=100)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

# 2. Définition de la Loss et de l'optimiseur
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(my_model.parameters(), lr=0.008, weight_decay=0.005)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.008,
    steps_per_epoch=len(my_train_loader),
    epochs=num_epochs,
    pct_start=0.3
)

start_time = time.time()
best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):
    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images, labels = images.to(device), labels.to(device)

        # Application de Mixup ou Cutmix sur le batch complet
        images, labels_mixed = cutmix_or_mixup(images, labels)

        optimizer.zero_grad()
        outputs = my_model(images)
        
        # Calcul de la perte avec les étiquettes mélangées
        loss = criterion(outputs, labels_mixed)
        
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    train_loss = running_loss / total_train

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = my_model(images)
            
            # La validation se fait sur les vraies étiquettes (sans mixup)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    # Sauvegarde du meilleur modèle
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    # Affichage régulier
    if (epoch + 1) % 5 == 0 or val_accuracy == best_val_accuracy:
        print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

# Restauration et sauvegarde finale
my_model.load_state_dict(best_model_weights)
torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time
print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")

Epoch [1/100] Train Loss: 4.4301 | Val Loss: 3.9864 | Val Acc: 8.74%
Epoch [2/100] Train Loss: 4.1945 | Val Loss: 3.6208 | Val Acc: 14.90%
Epoch [3/100] Train Loss: 4.0063 | Val Loss: 3.4379 | Val Acc: 17.56%
Epoch [4/100] Train Loss: 3.8949 | Val Loss: 3.3387 | Val Acc: 19.74%
Epoch [5/100] Train Loss: 3.8579 | Val Loss: 3.0649 | Val Acc: 23.92%
Epoch [6/100] Train Loss: 3.6848 | Val Loss: 3.0387 | Val Acc: 24.38%
Epoch [7/100] Train Loss: 3.6464 | Val Loss: 2.9436 | Val Acc: 26.22%
Epoch [8/100] Train Loss: 3.4945 | Val Loss: 2.6396 | Val Acc: 32.50%
Epoch [9/100] Train Loss: 3.4673 | Val Loss: 2.6214 | Val Acc: 33.14%
Epoch [10/100] Train Loss: 3.3764 | Val Loss: 2.3704 | Val Acc: 37.88%
Epoch [11/100] Train Loss: 3.3399 | Val Loss: 2.4215 | Val Acc: 38.00%
Epoch [13/100] Train Loss: 3.2126 | Val Loss: 2.1667 | Val Acc: 42.68%
Epoch [15/100] Train Loss: 3.1070 | Val Loss: 2.0889 | Val Acc: 44.40%
Epoch [16/100] Train Loss: 3.1011 | Val Loss: 2.0324 | Val Acc: 44.46%
Epoch [17/100] T

In [17]:
# Step 4 - Training and validation
import time
import copy
import torch
import torch.nn as nn
import random
from torchvision.transforms import v2

# Paramètres d'entraînement
num_epochs = 100

# 1. Initialisation de CutMix et Mixup pour 100 classes
cutmix = v2.CutMix(alpha=1.0, num_classes=100)
mixup = v2.MixUp(alpha=0.2, num_classes=100)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

# 2. Définition de la Loss standard (sans Label Smoothing pour laisser CutMix faire le travail)
criterion = nn.CrossEntropyLoss()

# 3. Optimiseur avec Weight Decay relâché (0.001) pour aider l'apprentissage final
optimizer = torch.optim.AdamW(my_model.parameters(), lr=0.008, weight_decay=0.001)

# 4. Planificateur OneCycleLR étiré sur 150 époques
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.008,
    steps_per_epoch=len(my_train_loader),
    epochs=num_epochs,
    pct_start=0.3
)

start_time = time.time()
best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):
    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images, labels = images.to(device), labels.to(device)

        # Application de Mixup ou Cutmix seulement 50% du temps
        if random.random() < 0.5:
            images, labels_mixed = cutmix_or_mixup(images, labels)
            loss_labels = labels_mixed
        else:
            loss_labels = labels

        optimizer.zero_grad()
        outputs = my_model(images)
        
        # Calcul de la perte avec les bonnes étiquettes (mélangées ou normales)
        loss = criterion(outputs, loss_labels)
        
        loss.backward()
        optimizer.step()
        
        # Mise à jour du scheduler à chaque itération (batch)
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    train_loss = running_loss / total_train

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = my_model(images)
            
            # La validation se fait toujours sur les vraies étiquettes
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    # Sauvegarde du meilleur modèle
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    # Affichage régulier (toutes les 5 époques ou s'il y a un nouveau record)
    if (epoch + 1) % 5 == 0 or val_accuracy == best_val_accuracy:
        print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

# Restauration et sauvegarde finale
my_model.load_state_dict(best_model_weights)
torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time
print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")

Epoch [1/100] Train Loss: 4.3526 | Val Loss: 3.8694 | Val Acc: 10.68%
Epoch [2/100] Train Loss: 4.0812 | Val Loss: 3.5809 | Val Acc: 15.58%
Epoch [3/100] Train Loss: 3.8972 | Val Loss: 3.3416 | Val Acc: 18.98%
Epoch [4/100] Train Loss: 3.6900 | Val Loss: 3.1238 | Val Acc: 23.12%
Epoch [5/100] Train Loss: 3.5316 | Val Loss: 3.2248 | Val Acc: 21.96%
Epoch [6/100] Train Loss: 3.4649 | Val Loss: 2.8361 | Val Acc: 27.80%
Epoch [7/100] Train Loss: 3.3458 | Val Loss: 2.6719 | Val Acc: 30.38%
Epoch [8/100] Train Loss: 3.2849 | Val Loss: 2.5856 | Val Acc: 32.98%
Epoch [10/100] Train Loss: 3.0180 | Val Loss: 2.5083 | Val Acc: 35.78%
Epoch [11/100] Train Loss: 2.9139 | Val Loss: 2.4020 | Val Acc: 37.22%
Epoch [13/100] Train Loss: 2.8356 | Val Loss: 2.2951 | Val Acc: 40.32%
Epoch [14/100] Train Loss: 2.7580 | Val Loss: 2.0211 | Val Acc: 45.66%
Epoch [15/100] Train Loss: 2.7327 | Val Loss: 1.9576 | Val Acc: 46.72%
Epoch [18/100] Train Loss: 2.6042 | Val Loss: 1.9254 | Val Acc: 47.34%
Epoch [19/100]

In [22]:
# Step 4 - Training and validation (SGD + Momentum + MultiStepLR)
import time
import copy
import torch
import torch.nn as nn
import random
from torchvision.transforms import v2

num_epochs = 150 

# 1. Initialisation de CutMix et Mixup (50% du temps)
cutmix = v2.CutMix(alpha=1.0, num_classes=100)
mixup = v2.MixUp(alpha=0.2, num_classes=100)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

# 2. Définition de la Loss standard
criterion = nn.CrossEntropyLoss()

# 3. NOUVEAU : Optimiseur SGD avec Momentum (Inspiré du TP)
# Un lr de 0.1 est le standard pour démarrer avec SGD sur ce type d'architecture
optimizer = torch.optim.SGD(my_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

# 4. NOUVEAU : MultiStepLR (Inspiré du TP)
# Le taux d'apprentissage sera divisé par 10 à l'époque 75, puis encore par 10 à l'époque 120
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[75, 120], gamma=0.1)

start_time = time.time()
best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):
    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images, labels = images.to(device), labels.to(device)

        # Application de Mixup ou Cutmix seulement 50% du temps
        if random.random() < 0.5:
            images, labels_mixed = cutmix_or_mixup(images, labels)
            loss_labels = labels_mixed
        else:
            loss_labels = labels

        optimizer.zero_grad()
        outputs = my_model(images)
        
        # Calcul de la perte avec les bonnes étiquettes (mélangées ou normales)
        loss = criterion(outputs, loss_labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    # NOUVEAU : Mise à jour du MultiStepLR à la fin de l'époque (et non plus dans le batch)
    scheduler.step()

    train_loss = running_loss / total_train

    # ----- VALIDATION -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = my_model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    # Sauvegarde du meilleur modèle
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    # Affichage de l'époque, et on ajoute le Learning Rate actuel pour vérifier le changement aux époques 75 et 120
    current_lr = optimizer.param_groups[0]['lr']
    if (epoch + 1) % 5 == 0 or val_accuracy == best_val_accuracy:
        print(f"Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.5f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

# Restauration et sauvegarde finale
my_model.load_state_dict(best_model_weights)
torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time
print("\nTraining completed!")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")

Epoch [1/150] LR: 0.10000 | Train Loss: 4.2933 | Val Loss: 3.7934 | Val Acc: 11.22%
Epoch [2/150] LR: 0.10000 | Train Loss: 4.0005 | Val Loss: 3.5130 | Val Acc: 15.74%
Epoch [3/150] LR: 0.10000 | Train Loss: 3.8045 | Val Loss: 3.2457 | Val Acc: 20.90%
Epoch [4/150] LR: 0.10000 | Train Loss: 3.6020 | Val Loss: 3.0138 | Val Acc: 23.68%
Epoch [5/150] LR: 0.10000 | Train Loss: 3.4299 | Val Loss: 3.0068 | Val Acc: 24.30%
Epoch [6/150] LR: 0.10000 | Train Loss: 3.3506 | Val Loss: 2.6711 | Val Acc: 30.98%
Epoch [7/150] LR: 0.10000 | Train Loss: 3.2198 | Val Loss: 2.5920 | Val Acc: 33.64%
Epoch [9/150] LR: 0.10000 | Train Loss: 2.9700 | Val Loss: 2.3206 | Val Acc: 38.94%
Epoch [10/150] LR: 0.10000 | Train Loss: 2.9010 | Val Loss: 2.3661 | Val Acc: 38.56%
Epoch [12/150] LR: 0.10000 | Train Loss: 2.7289 | Val Loss: 2.2052 | Val Acc: 41.82%
Epoch [13/150] LR: 0.10000 | Train Loss: 2.7435 | Val Loss: 2.1251 | Val Acc: 43.56%
Epoch [14/150] LR: 0.10000 | Train Loss: 2.6874 | Val Loss: 2.0510 | Val 

In [25]:
 # Step 4 - Training and validation (SGD + Momentum + MultiStepLR + SWA)
import time
import copy
import torch
import torch.nn as nn
import random
from torchvision.transforms import v2

# NOUVEAU : Importation des outils SWA
from torch.optim.swa_utils import AveragedModel, update_bn

num_epochs = 150 
swa_start = 120  # L'époque à partir de laquelle on commence à moyenner les poids

# Initialisation de CutMix et Mixup (50% du temps)
cutmix = v2.CutMix(alpha=1.0, num_classes=100)
mixup = v2.MixUp(alpha=0.2, num_classes=100)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])

# Définition de la Loss standard
criterion = nn.CrossEntropyLoss()

# Optimiseur SGD avec Momentum
optimizer = torch.optim.SGD(my_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

# MultiStepLR
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[75, 120], gamma=0.1)

# NOUVEAU : Création du modèle qui va stocker la moyenne des poids
swa_model = AveragedModel(my_model)

start_time = time.time()
best_val_accuracy = 0.0
best_model_weights = copy.deepcopy(my_model.state_dict())

for epoch in range(num_epochs):
    # ----- TRAINING -----
    my_model.train()
    running_loss = 0.0
    total_train = 0

    for images, labels in my_train_loader:
        images, labels = images.to(device), labels.to(device)

        if random.random() < 0.5:
            images, labels_mixed = cutmix_or_mixup(images, labels)
            loss_labels = labels_mixed
        else:
            loss_labels = labels

        optimizer.zero_grad()
        outputs = my_model(images)
        
        loss = criterion(outputs, loss_labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total_train += labels.size(0)

    # Mise à jour du MultiStepLR à la fin de l'époque
    scheduler.step()

    # NOUVEAU : On met à jour la moyenne des poids pour le modèle SWA
    if epoch >= swa_start:
        swa_model.update_parameters(my_model)

    train_loss = running_loss / total_train

    # ----- VALIDATION (sur le modèle standard pour suivre l'évolution) -----
    my_model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in my_val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = my_model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = running_val_loss / total
    val_accuracy = 100 * correct / total

    # Sauvegarde du meilleur modèle standard
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_weights = copy.deepcopy(my_model.state_dict())

    current_lr = optimizer.param_groups[0]['lr']
    if (epoch + 1) % 5 == 0 or val_accuracy == best_val_accuracy:
        print(f"Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.5f} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

# ----- ÉVALUATION FINALE AVEC SWA -----
print("\n--- Mise à jour de la Batch Normalization pour SWA ---")
# Essentiel : SWA doit recalculer les statistiques de la BatchNorm sur les données d'entraînement
update_bn(my_train_loader, swa_model, device=device)

print("--- Évaluation du modèle SWA final ---")
swa_model.eval()
swa_correct = 0
swa_total = 0

with torch.no_grad():
    for images, labels in my_val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = swa_model(images)
        _, predicted = torch.max(outputs, 1)
        swa_total += labels.size(0)
        swa_correct += (predicted == labels).sum().item()

swa_accuracy = 100 * swa_correct / swa_total
print(f"Validation Accuracy avec SWA : {swa_accuracy:.2f}%")

# On sauvegarde le meilleur des deux modèles (le standard ou le SWA)
if swa_accuracy > best_val_accuracy:
    print("Le modèle SWA est meilleur ! Sauvegarde de neural_net.pt avec les poids SWA.")
    # On sauvegarde le module interne du AveragedModel pour que les poids soient compatibles
    torch.save(swa_model.module.state_dict(), "neural_net.pt")
else:
    print("Le modèle standard était meilleur. Restauration de neural_net.pt avec le meilleur modèle standard.")
    my_model.load_state_dict(best_model_weights)
    torch.save(my_model.state_dict(), "neural_net.pt")

elapsed_time = time.time() - start_time
print(f"\nTraining time: {elapsed_time / 60:.2f} minutes")

Epoch [1/150] LR: 0.10000 | Train Loss: 4.2933 | Val Loss: 3.7934 | Val Acc: 11.22%
Epoch [2/150] LR: 0.10000 | Train Loss: 4.0005 | Val Loss: 3.5130 | Val Acc: 15.74%
Epoch [3/150] LR: 0.10000 | Train Loss: 3.8045 | Val Loss: 3.2457 | Val Acc: 20.90%
Epoch [4/150] LR: 0.10000 | Train Loss: 3.6020 | Val Loss: 3.0138 | Val Acc: 23.68%
Epoch [5/150] LR: 0.10000 | Train Loss: 3.4299 | Val Loss: 3.0068 | Val Acc: 24.30%
Epoch [6/150] LR: 0.10000 | Train Loss: 3.3506 | Val Loss: 2.6711 | Val Acc: 30.98%
Epoch [7/150] LR: 0.10000 | Train Loss: 3.2198 | Val Loss: 2.5920 | Val Acc: 33.64%
Epoch [9/150] LR: 0.10000 | Train Loss: 2.9700 | Val Loss: 2.3206 | Val Acc: 38.94%
Epoch [10/150] LR: 0.10000 | Train Loss: 2.9010 | Val Loss: 2.3661 | Val Acc: 38.56%
Epoch [12/150] LR: 0.10000 | Train Loss: 2.7289 | Val Loss: 2.2052 | Val Acc: 41.82%
Epoch [13/150] LR: 0.10000 | Train Loss: 2.7435 | Val Loss: 2.1251 | Val Acc: 43.56%
Epoch [14/150] LR: 0.10000 | Train Loss: 2.6874 | Val Loss: 2.0510 | Val 